In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install keras


In [ ]:
import os
import pandas as pd
import numpy as np


from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing import sequence

from tensorflow.python.keras.utils.np_utils import to_categorical
from keras.callbacks import EarlyStopping
from tqdm import tqdm, trange

#from keras.layers.Wrapper import Bidirectional


# from keras.preprocessing.text import Tokenizer

In [ ]:
# !pip install transformers==4.31.0
!pip install transformers

In [ ]:
import tensorflow as tf
physical_devices = tf.config.list_physical_devices('GPU')
# tf.config.experimental.set_memory_growth(physical_devices[0], enable=True)

In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Fake News/test.csv")
#data = pd.read_csv("/content/drive/MyDrive/Dhaka Metro Rail comment to sentiment/MetroRail FInal Data.csv", encoding='latin1')

# Check the list of column names in the DataFrame
print(data.columns)

# Drop the column if it exists
column_to_drop = 'Unnamed: 0'
if column_to_drop in data.columns:
    data = data.drop(columns=column_to_drop)

data

Index(['ID', 'label', 'statement', 'subject', 'speaker', 'speaker_job',
       'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts',
       'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts',
       'context', 'sentiment', 'sentiment_score', 'sentiment_magnitude',
       'anger', 'fear', 'joy', 'disgust', 'sad', 'speaker_id', 'list',
       'sentiment_code'],
      dtype='object')


,ID,label,statement,subject,speaker,speaker_job,state_info,party_affiliation,barely_true_counts,false_counts,...,sentiment_score,sentiment_magnitude,anger,fear,joy,disgust,sad,speaker_id,list,sentiment_code
0,11972.json,true,Building a wall on the U.S.-Mexico border will...,immigration,rick-perry,Governor,Texas,republican,30,30,...,-0.2,0.2,0.067151,0.155968,0.368879,0.198711,0.311238,_0_,"[1, 0]",_NEG_
1,11685.json,false,Wisconsin is on pace to double the number of l...,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,...,0.0,0.0,0.050274,0.054154,0.195262,0.069050,0.287632,_1_,"[0, 1]",NaN
2,11096.json,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,...,-0.8,0.8,0.055874,0.199553,0.115140,0.439826,0.438706,_2_,"[0, 1]",_NEG_
3,5209.json,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,...,0.0,0.0,0.158225,0.107996,0.289932,0.068629,0.368779,_3_,"[0, 1]",NaN
4,9524.json,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,...,-0.7,0.7,0.128335,0.119912,0.082317,0.299316,0.283066,_4_,"[0, 1]",_NEG_
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1262,7334.json,half-true,Says his budget provides the highest state fun...,education,rick-scott,Governor,Florida,republican,28,23,...,0.3,0.3,0.089693,0.249144,0.162474,0.214786,0.211747,_32_,"[0, 1]",_POS_
1263,9788.json,barely-true,Ive been here almost every day.,"civil-rights,crime,criminal-justice",jay-nixon,Governor,Missouri,democrat,2,0,...,0.3,0.3,0.086518,0.222013,0.237928,0.025325,0.507414,_441_,"[0, 1]",_POS_
1264,10710.json,barely-true,"In the early 1980s, Sen. Edward Kennedy secret...","bipartisanship,congress,foreign-policy,history",mackubin-thomas-owens,"senior fellow, Foreign Policy Research Institute",Rhode Island,columnist,1,0,...,0.1,0.1,0.318661,0.080356,0.156447,0.088780,0.165929,_634_,"[0, 1]",_POS_
1265,3186.json,barely-true,Says an EPA permit languished under Strickland...,"environment,government-efficiency",john-kasich,"Governor of Ohio as of Jan. 10, 2011",Ohio,republican,9,8,...,-0.2,0.2,0.135285,0.062356,0.333818,0.079994,0.201202,_172_,"[0, 1]",_NEG_


In [ ]:
from tensorflow.keras import layers
from tensorflow import keras
import tensorflow as tf

from sklearn.model_selection import train_test_split
from ast import literal_eval

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

In [ ]:
data.shape

(1267, 25)

In [ ]:
character_set_not =  ['\n',
 '!',
 '"',
 '#',
 '&',
 "'",
 '(',
 ')',
 '*',
 '+',
 ',',
 '-',
 '.',
 '/',
 ':',
 ';',
 '=',
 '?',
 '[',
 '\\',
 ']',
 '{',
 '}',
 '\xa0',
 '¬',
 '´',
 '·',
 '।',
 '\u09e4',
 '৷',
 '\u200d',
 '–',
 '—',
 '‘',
 '’',
 '“',
 '”',
 '•']

# Handling Stopwords

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words_list= list(stop_words)
print(stop_words_list)


['my', 'here', 'she', 'who', 'for', "hasn't", 'off', 'm', 'should', 'if', 'because', "hadn't", 'an', 'not', 'until', "wouldn't", "won't", "that'll", 'where', 'each', 'myself', 'don', 'y', 'they', 'only', "doesn't", 'any', 'himself', 'that', 'under', 'ours', "mustn't", 'doing', 'aren', 'was', 'there', 'our', 'over', 'hers', "you're", 'these', 'mustn', 'him', 'during', "it's", 'whom', 'up', 'have', 'about', 'do', 's', 'both', "you'll", 'is', 'as', 'we', 'am', 'to', 'his', 'when', 'hadn', 'needn', "needn't", 'be', 'has', 'before', "didn't", 'o', 'mightn', 'too', 'at', "isn't", 'from', 'out', 'few', 'can', 'than', 'did', 'again', 'above', 'most', 'those', 'same', "you'd", 'after', 'further', 'couldn', 'of', "aren't", 'the', 'you', 'nor', 'why', "she's", 'ma', 'your', 'them', 'a', 'this', 'shan', 'been', 'd', 'themselves', "weren't", 'are', 'and', 'by', 'against', 'yours', 'he', 'below', "shan't", 'yourselves', 'i', 'won', "haven't", 'yourself', 'herself', 'but', 'on', 'now', 'more', 'while

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
for i in stop_words_list:
  stop_words_list[stop_words_list.index(i)]=' '+i.replace(' ','')+' '
print(stop_words_list)

[' my ', ' here ', ' she ', ' who ', ' for ', " hasn't ", ' off ', ' m ', ' should ', ' if ', ' because ', " hadn't ", ' an ', ' not ', ' until ', " wouldn't ", " won't ", " that'll ", ' where ', ' each ', ' myself ', ' don ', ' y ', ' they ', ' only ', " doesn't ", ' any ', ' himself ', ' that ', ' under ', ' ours ', " mustn't ", ' doing ', ' aren ', ' was ', ' there ', ' our ', ' over ', ' hers ', " you're ", ' these ', ' mustn ', ' him ', ' during ', " it's ", ' whom ', ' up ', ' have ', ' about ', ' do ', ' s ', ' both ', " you'll ", ' is ', ' as ', ' we ', ' am ', ' to ', ' his ', ' when ', ' hadn ', ' needn ', " needn't ", ' be ', ' has ', ' before ', " didn't ", ' o ', ' mightn ', ' too ', ' at ', " isn't ", ' from ', ' out ', ' few ', ' can ', ' than ', ' did ', ' again ', ' above ', ' most ', ' those ', ' same ', " you'd ", ' after ', ' further ', ' couldn ', ' of ', " aren't ", ' the ', ' you ', ' nor ', ' why ', " she's ", ' ma ', ' your ', ' them ', ' a ', ' this ', ' shan 

In [ ]:
def tokenize(s):
    for i in character_set_not:
        s =  s.replace(i, '')
    for j in stop_words_list:
        s= s.replace(j, ' ')
    s=s.lower()
    return  s

In [ ]:
data['statement'] = data['statement'].fillna('').map(tokenize)
data

,ID,label,statement,subject,speaker,speaker_job,state_info,party_affiliation,barely_true_counts,false_counts,...,sentiment_score,sentiment_magnitude,anger,fear,joy,disgust,sad,speaker_id,list,sentiment_code
0,11972.json,true,building wall usmexico border take literally y...,immigration,rick-perry,Governor,Texas,republican,30,30,...,-0.2,0.2,0.067151,0.155968,0.368879,0.198711,0.311238,_0_,"[1, 0]",_NEG_
1,11685.json,false,wisconsin pace double number layoffs year,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,...,0.0,0.0,0.050274,0.054154,0.195262,0.069050,0.287632,_1_,"[0, 1]",NaN
2,11096.json,false,says john mccain done nothing help vets,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,...,-0.8,0.8,0.055874,0.199553,0.115140,0.439826,0.438706,_2_,"[0, 1]",_NEG_
3,5209.json,half-true,suzanne bonamici supports plan cut choice medi...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,...,0.0,0.0,0.158225,0.107996,0.289932,0.068629,0.368779,_3_,"[0, 1]",NaN
4,9524.json,pants-fire,when asked reporter whether hes center crimina...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,...,-0.7,0.7,0.128335,0.119912,0.082317,0.299316,0.283066,_4_,"[0, 1]",_NEG_
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1262,7334.json,half-true,says budget provides highest state funding lev...,education,rick-scott,Governor,Florida,republican,28,23,...,0.3,0.3,0.089693,0.249144,0.162474,0.214786,0.211747,_32_,"[0, 1]",_POS_
1263,9788.json,barely-true,ive almost every day,"civil-rights,crime,criminal-justice",jay-nixon,Governor,Missouri,democrat,2,0,...,0.3,0.3,0.086518,0.222013,0.237928,0.025325,0.507414,_441_,"[0, 1]",_POS_
1264,10710.json,barely-true,in early 1980s sen edward kennedy secretly off...,"bipartisanship,congress,foreign-policy,history",mackubin-thomas-owens,"senior fellow, Foreign Policy Research Institute",Rhode Island,columnist,1,0,...,0.1,0.1,0.318661,0.080356,0.156447,0.088780,0.165929,_634_,"[0, 1]",_POS_
1265,3186.json,barely-true,says epa permit languished strickland new epa ...,"environment,government-efficiency",john-kasich,"Governor of Ohio as of Jan. 10, 2011",Ohio,republican,9,8,...,-0.2,0.2,0.135285,0.062356,0.333818,0.079994,0.201202,_172_,"[0, 1]",_NEG_


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Flatten, Activation
from tensorflow.keras.models import Model
from transformers import TFBertModel, BertConfig



In [ ]:
import pandas as pd



df = pd.DataFrame(data)

# Find maximum and minimum character count for the 'News' column
max_char_count = df['statement'].apply(len).max()
min_char_count = df['statement'].apply(len).min()

# Find maximum and minimum word count for the 'News' column
max_word_count = df['statement'].apply(lambda x: len(x.split())).max()
min_word_count = df['statement'].apply(lambda x: len(x.split())).min()

print("Maximum Character Count:", max_char_count)
print("Minimum Character Count:", min_char_count)
print("Maximum Word Count:", max_word_count)
print("Minimum Word Count:", min_word_count)


Maximum Character Count: 2474
Minimum Character Count: 11
Maximum Word Count: 327
Minimum Word Count: 2


In [ ]:
max_length=296
num_labels=6
batch_size=24

# Model

In [ ]:
# import tensorflow as tf
# from transformers import TFBertModel, BertTokenizer

# # Load pre-trained BERT model and tokenizer
# bert_model = TFBertModel.from_pretrained('bert-base-multilingual-uncased')
# # from transformers import AutoTokenizer

# # bert_model = AutoTokenizer.from_pretrained('bert-base-cased')

# # Define input layers
# input_ids = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='input_ids')
# attention_mask = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='attention_mask')
# token_type_ids = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='token_type_ids')

# # Encode input using BERT model
# bert_output = bert_model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

# # Get pooled output and pass through dropout layer
# pooled_output = tf.keras.layers.Dropout(0.1)(bert_output.pooler_output)

# # Add dense layer for classification
# num_labels = 6 # number of unique labels
# outputs = Dense(6, use_bias=False)(pooled_output)
# #class_probs = tf.keras.layers.Activation(keras.activations.sigmoid)(outputs)
# class_probs = tf.keras.layers.Activation(keras.activations.softmax)(outputs)

# # Define the model inputs and outputs
# model_inputs = [input_ids, attention_mask, token_type_ids]
# model_outputs = class_probs

# # Build the model
# model = tf.keras.models.Model(inputs=model_inputs, outputs=model_outputs)

# # Compile the model
# optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-08, clipnorm=1.0)
# loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# #loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
# #metric = tf.keras.metrics.BinaryAccuracy('accuracy')

# model.compile(optimizer=optimizer, loss=loss, metrics=[metric])


In [ ]:
# !pip install --upgrade tensorflow transformers


## OKay

In [ ]:
# import tensorflow as tf
# from transformers import TFBertModel, BertTokenizer
# from tensorflow.keras.layers import Layer, Dense

# # Define max_length
# max_length = 128  # Example value, adjust as needed

# # Load pre-trained BERT model and tokenizer
# bert_model = TFBertModel.from_pretrained('bert-base-uncased')
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# # Custom layer to wrap the BERT model
# class BertLayer(Layer):
#     def __init__(self, bert_model, **kwargs):
#         super(BertLayer, self).__init__(**kwargs)
#         self.bert = bert_model

#     def call(self, inputs):
#         input_ids, attention_mask, token_type_ids = inputs
#         outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
#         # Return the pooler_output tensor
#         return outputs.pooler_output

# # Define input layers as KerasTensors
# input_ids = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='input_ids')
# attention_mask = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='attention_mask')
# token_type_ids = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='token_type_ids')

# # Use the custom layer to call the BERT model
# bert_outputs = BertLayer(bert_model)([input_ids, attention_mask, token_type_ids])

# # Add dense layer for classification (without softmax)
# num_labels = 6  # Number of unique labels
# dense_layer = Dense(num_labels, use_bias=False)(bert_outputs)

# # Build the model using Keras Functional API
# model = tf.keras.Model(inputs=[input_ids, attention_mask, token_type_ids], outputs=dense_layer)

# # Print the model summary
# model.summary()

# # Compile the model with from_logits=True
# optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-08, clipnorm=1.0)
# loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
# model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

# # The model is now ready to be trained


In [ ]:
import tensorflow as tf
from transformers import TFBertModel, BertTokenizer
from tensorflow.keras.layers import Layer, Dense

# Define max_length
max_length = 128  # Example value, adjust as needed

# Load pre-trained BERT model and tokenizer
bert_model = TFBertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


        class BertLayer(Layer):
    def __init__(self, bert_model, **kwargs):
        super(BertLayer, self).__init__(**kwargs)
        self.bert = bert_model

    def call(self, inputs):
        input_ids, attention_mask, token_type_ids = inputs
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        # Return the pooler_output tensor
        return outputs.pooler_output

    def get_config(self):
        config = super().get_config()
        # Add any necessary configuration parameters here, e.g.,
        # config.update({
        #     "param1": self.param1,
        #     "param2": self.param2
        # })
        return config


# Define input layers as KerasTensors
input_ids = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='input_ids')
attention_mask = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='attention_mask')
token_type_ids = tf.keras.layers.Input(shape=(max_length,), dtype=tf.int32, name='token_type_ids')

# Use the custom layer to call the BERT model
bert_outputs = BertLayer(bert_model)([input_ids, attention_mask, token_type_ids])

# Add dense layer for classification
num_labels = 6  # Number of unique labels
dense_layer = Dense(num_labels, use_bias=False)(bert_outputs)
class_probs = tf.keras.layers.Activation(tf.keras.activations.softmax)(dense_layer)

# Build the model using Keras Functional API
model = tf.keras.Model(inputs=[input_ids, attention_mask, token_type_ids], outputs=class_probs)

# Print the model summary
model.summary()


IndentationError: unindent does not match any outer indentation level (<tokenize>, line 14)

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-08, clipnorm=1.0)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [ ]:
unique_labels = data['label'].unique()

In [ ]:
for label in unique_labels:
    count = df[df['label'] == label].shape[0]
    print(f"Label: {label}, Count: {count}")

In [ ]:
for label in unique_labels:
    count = data[data['label'] == label].shape[0]
    print(f"Label: {label}, Count: {count}")

In [ ]:
labels = ['half-true', 'mostly-true', 'false', 'true', 'barely-true','pants-fire' ]
labels

In [ ]:
def convert(label):
    return labels.index(label)


In [ ]:
def convert(label):
    try:
        return labels.index(label)
    except ValueError:
        return -1  # or any other default value to indicate that the label is not found


In [ ]:
data['label']=data['label'].map(convert)
data

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Load your dataset and split it into training and validation sets

X_train, X_val, y_train, y_val = train_test_split(data['statement'], data['label'], test_size=0.2, random_state=42)

y_val


In [ ]:
X_train

In [ ]:
from transformers import BertTokenizer
import numpy as np

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')

# Tokenize the input data
X_train_tokens = tokenizer.batch_encode_plus(X_train.values, padding=True, truncation=True, max_length=max_length, add_special_tokens=True)
X_val_tokens = tokenizer.batch_encode_plus(X_val, padding=True, truncation=True, max_length=max_length , add_special_tokens=True)

In [ ]:
X_train_tokens

In [ ]:
# Convert the token IDs, attention mask, and label data to numpy arrays
X_train_input_ids = np.array(X_train_tokens['input_ids'])
X_train_attention_mask = np.array(X_train_tokens['attention_mask'])
X_train_token_type_id = np.array(X_train_tokens['token_type_ids'])
X_val_input_ids = np.array(X_val_tokens['input_ids'])
X_val_attention_mask = np.array(X_val_tokens['attention_mask'])
X_val_token_type_id = np.array(X_val_tokens['token_type_ids'])
#y_train = np.asarray(y_train).astype(np.float32)
#y_val = np.array(y_val)

In [ ]:
def creat_input(inputIds, attenMask, TTIds):
  x=[
      inputIds,
      attenMask,
      TTIds
  ]
  return x

In [ ]:
x_train= creat_input(X_train_input_ids, X_train_attention_mask, X_train_token_type_id)
x_val= creat_input(X_val_input_ids, X_val_attention_mask, X_val_token_type_id)


In [ ]:
y_train= np.array(y_train.tolist())
y_train = tf.convert_to_tensor(y_train)#, dtype=tf.int64)
y_val= np.array(y_val.tolist())
y_val = tf.convert_to_tensor(y_val)



In [ ]:
y_train

In [ ]:
y_val

In [ ]:
X_train_input_ids.shape



In [ ]:
# Check the shapes of your data and labels
print("Shape of x_train:", x_train)
print("Shape of y_train:", y_train)  # Make sure you have y_train defined
print("Shape of x_val:", x_val)
print("Shape of y_val:", y_val)

## **Model Training**

In [ ]:
#from tensorflow.keras.callbacks import ModelCheckpoint

# Define a filepath for the saved model
#model_filepath = 'Compound to sentiment/model1'

# Define the ModelCheckpoint callback to save the model with the best validation accuracy
#checkpoint_callback = ModelCheckpoint(model_filepath, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)

# Train the model
history = model.fit(
    x=x_train,
    y=y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=batch_size,
   # callbacks=[checkpoint_callback]

)

# Load the saved model with the best validation accuracy
#model.load_weights(model_filepath)

ValueError: Data cardinality is ambiguous. Make sure all arrays contain the same number of samples.'x' sizes: 3
'y' sizes: 1013


In [ ]:
res=pd.DataFrame(history.history)
res.to_csv("/content/drive/MyDrive/Padma Bridge/accuracy_loss.csv")

In [ ]:
model.summary()


# **Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix
#model_filepath = '/content/drive/MyDrive/Dhaka Metro Rail comment to sentiment'

X_val = {
    'input_ids': X_val_input_ids,
    'attention_mask': X_val_attention_mask,
    'token_type_ids': X_val_token_type_id
}
# Get the predicted probabilities for the validation set
y_pred = model.predict([X_val['input_ids'], X_val['attention_mask'], X_val['token_type_ids']])

# Convert the predicted probabilities to binary labels
#y_pred_binary = (y_pred > 0.4).astype(int)
y_classes = y_pred.argmax(axis=-1)

# Define a list of all possible class labels
labels = [i for i in range(num_labels)]

# Calculate the confusion matrix
labels = [0, 1, 2]
#cm = confusion_matrix(y_val, y_pred_binary, labels=labels)
#labels = [0, 1]
cm = confusion_matrix(y_val, y_classes, labels=labels)
# Print the confusion matrix
print(cm)


# Calculate TP FP FN TN & Precision..

In [ ]:
# Get the confusion matrix

cm = confusion_matrix(y_val, y_classes, labels=labels)
# Calculate true positives, false positives, false negatives, and true negatives for each class
TP = np.diag(cm)
FP = np.sum(cm, axis=0) - TP
FN = np.sum(cm, axis=1) - TP
TN = np.sum(cm) - (TP + FP + FN)

# Print the results
for i in range(num_labels):
    print(f"Class {i} - TP: {TP[i]}, FP: {FP[i]}, FN: {FN[i]}, TN: {TN[i]}")


In [ ]:
FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)

TPR = TP / (TP + FN)   #Recall
TNR = TN / (TN + FP)
PPV = TP / (TP + FP)  #precision
NPV = TN / (TN + FN)
FPR = FP / (FP + TN)
FNR = FN / (TP + FN)
FDR = FP / (TP + FP)
ACC = (TP + TN) / (TP + FP + FN + TN) #Accuracy
FI= 2*(PPV * TPR)/ (PPV + TPR)
# Macro F1 Score is the average F1 Score across all classes
Macro_F1 = np.mean(FI)

print('False Positives:', FP)
print('False Negatives:', FN)
print('True Positives:', TP)
print('True Negatives:', TN)
print('Sensitivity (True Positive Rate) / Recall:', TPR)
print('Specificity (True Negative Rate):', TNR)
print('Precision:', PPV)
print('Negative Predictive Value:', NPV)
print('False Positive Rate:', FPR)
print('False Negative Rate:', FNR)
print('False Discovery Rate:', FDR)
print('Overall Accuracy:', ACC)
print('FI Score:', FI)
print('Macro F1 Score:', Macro_F1)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
cm=[[63, 10,  8],
    [3, 29,  15],
    [ 4,  7, 48]]
cmap = "flare"
text_labels = ['positive', 'negative', 'neutral']
sns.heatmap(data = cm, cmap=cmap, annot=True,linewidth=.5, linecolor='Black',xticklabels=text_labels, yticklabels=text_labels, fmt="d")
plt.savefig('/content/drive/MyDrive/Elevated Expressway/Confusion Matrix.pdf', dpi=300, transparent=True,bbox_inches='tight')
plt.show()